In [1]:
import os

import pandas as pd
import torch
from torch.utils.data import DataLoader

from train_uhwr import evaluate
from tokeniser import get_tokenizer
from model.joint_model import JointModel
from utils.dataset import HWRDataset, collate_fn


In [ ]:
ROOT = "<CODE_ROOT_PATH>"
DATA_ROOT = "<DATA_ROOT_PATH>"
os.chdir(ROOT)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# train_df = pd.read_csv(f"{DATA_ROOT}/data_uhwr/train.csv")
# train_df = pd.read_csv(f"{DATA_ROOT}/data_upti/train_touse.csv")
# eval_df = pd.read_csv(f"{DATA_ROOT}/data_uhwr/val.csv")
eval_df = pd.read_csv(f"{DATA_ROOT}/data_uhwr/test.csv")

tokenizer = get_tokenizer()

# train_dataset = HWRDataset(DATA_ROOT, train_df, tokenizer, aug=True)
eval_dataset = HWRDataset(DATA_ROOT, eval_df, tokenizer)

# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                            # collate_fn=collate_fn, num_workers=4)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False,
                            collate_fn=collate_fn, num_workers=4)

model = JointModel(
    trans_enc_d_model=256,
    trans_enc_nhead=8,
    trans_enc_layers=3,
    trans_enc_ff_dim=1024,
    tokenizer=tokenizer,
    trans_dec_d_model=256,
    trans_dec_nhead=8,
    trans_dec_layers=3,
    trans_dec_n_positions=512,
    freeze_decoder=True,   # 🔥 paper-aligned
    # freeze_decoder=False,
    decoder_path="decoder_pretrain_tokenizer_bos_eos\\checkpoint-32452"
).to(device)

model.load_state_dict(torch.load("best_model_uhwr_icdar.pt"))

<All keys matched successfully>

In [4]:
val_loss, val_cer = evaluate(
        model, eval_loader, tokenizer, device, None, decode_mode='beam_search', loss_calc=False
    )

Evaluating:   0%|          | 0/17 [00:00<?, ?it/s]c:\ProgramData\miniconda3\envs\uhwr\Lib\site-packages\transformers\generation\utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(
Evaluating: 100%|██████████| 17/17 [00:44<00:00,  2.60s/it]


In [5]:
val_cer*100

6.460404614728038